# How This Blog Is Built — figures

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josephrich98/joseph_rich_blog/blob/main/posts/how-this-blog-is-built/notebook.ipynb)

This post is an explainer about the blog's own toolchain, so it ships **no data**. The single figure is a schematic of the commit-to-deploy pipeline, drawn entirely from box/arrow geometry so it is fully reproducible. Running this notebook top-to-bottom regenerates `figures/pipeline.png`, which `main.md` embeds. The same code lives standalone in [`scripts/make_figures.py`](scripts/make_figures.py).

In [ ]:
# NBVAL_IGNORE_OUTPUT
import os

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch

FIG = "figures"
os.makedirs(FIG, exist_ok=True)

# Two colors carry the whole legend: blue for the steps I trigger by hand,
# black for everything that then happens automatically.
BLUE = "#1565c0"   # manual: writing + the git commands I run
BLACK = "#1b1b1b"  # automatic: whatever those git commands set off

In [ ]:
def box(ax, xy, w, h, text, *, fontsize=11, text_color=BLACK):
    """Draw a rounded white box (black outline) centered at xy with text."""
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.08",
        linewidth=1.6, edgecolor=BLACK, facecolor="white", zorder=2,
    ))
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            color=text_color, zorder=3, linespacing=1.35)


def arrow(ax, p0, p1, color=BLACK, style="-|>", lw=1.8, ls="-",
          connectionstyle="arc3,rad=0"):
    ax.add_patch(FancyArrowPatch(
        p0, p1, arrowstyle=style, mutation_scale=14, linewidth=lw,
        linestyle=ls, color=color, shrinkA=2, shrinkB=2, zorder=1,
        connectionstyle=connectionstyle,
    ))

In [ ]:
# NBVAL_IGNORE_OUTPUT
fig, ax = plt.subplots(figsize=(9.0, 6.6))
ax.set_xlim(0, 9.0)
ax.set_ylim(0, 6.6)
ax.axis("off")

cx = 3.3     # the three blocks share one vertical spine
w = 6.0      # wide enough for the longest label

# --- Block 1: write it (manual) ----------------------------------------------
box(ax, (cx, 5.7), w, 0.9,
    "Write $\tt main.md$ (+ $\tt notebook.ipynb$)", text_color=BLUE)

# git commit -------------------------------------------------------------------
arrow(ax, (cx, 5.25), (cx, 4.30))
ax.text(cx + 0.25, 4.78, "git commit", ha="left", va="center",
        fontsize=10.5, color=BLUE, style="italic")

# --- Block 2: sync the site + check the citations (automatic) ----------------
box(ax, (cx, 3.75), w, 1.1,
    "update site with Jekyll + Vercel\n"
    "validate updated references with doi2bib")

# git push ---------------------------------------------------------------------
arrow(ax, (cx, 3.20), (cx, 2.25))
ax.text(cx + 0.25, 2.72, "git push", ha="left", va="center",
        fontsize=10.5, color=BLUE, style="italic")

# --- Block 3: deploy + test (automatic) --------------------------------------
box(ax, (cx, 1.70), w, 1.1,
    "deploy site\n"
    "run notebook checks on updated notebooks (strict → lax)")

# GitHub Actions re-runs the notebook checks on a schedule (self-loop).
edge = cx + w / 2
arrow(ax, (edge, 1.95), (edge, 1.45), connectionstyle="arc3,rad=-2.2")
ax.text(edge + 0.9, 1.70, "GitHub actions (6mo)", ha="left", va="center",
        fontsize=10, color=BLACK, style="italic")

fig.tight_layout()
fig.savefig(os.path.join(FIG, "pipeline.png"), dpi=200, bbox_inches="tight")
plt.show()